In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Lab: Deep-Dive into Google Colab


|    Group Members      | Student Id |
|:----------------------|:-----------:|
| Dinal Maha Vidanelage | 2425591 |
| Olga Chitembo         | 2412122 |
| Sandip Ranjit         | 2424444    |
| Twe He Gam            | 2412112|

*Note: To prevent code overwrites and merge conflicts, one member created the primary master notebook and everyone worked on a cloned copy to test edge cases (such as GPU memory limits and precision checks).*

In [ ]:
x = 10
print(x)

10


In [ ]:
!pwd

/content


In [ ]:
!ls

drive  sample_data


In [ ]:
%cd /content/drive/MyDrive/Data Handling and Machine Learning

/content/drive/MyDrive/Data Handling and Machine Learning


In [ ]:
import os

print("Python version:")
print(os.sys.version)

print("\nCurrent working directory:")
print(os.getcwd())

Python version:
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Current working directory:
/content/drive/.shortcut-targets-by-id/1wIXBoTAvdbYOXIGoJehJOd8zm1gA7wE2/Data Handling and Machine Learning


# Task 1A - Working Directories: The Classic !cd vs %cd Experiment

**Prediction**

What do you expect this to print?
```
!cd /tmp
```
followed by, in a new cell:
```
!pwd
```

- As a first time user of Google Colab, I would expect that `!cd /tmp` would change the directory and output `/tmp`.

In [ ]:
!cd /tmp

In [ ]:
!pwd

/content/drive/MyDrive/Data Handling and Machine Learning


In [ ]:
%cd /tmp

/tmp


In [ ]:
!pwd

/tmp


In [ ]:
import os
print(os.getcwd())

/tmp


# Task 1B - Important Counterexample: ! Can Change the Filesystem

In [ ]:
!mkdir -p my_project
!touch my_project/data.txt

In [ ]:
!ls -l my_project

total 0
-rw-r--r-- 1 root root 0 Aug 19 13:37 data.txt


# Task 1C - Shell Variables vs Python Variables

**Experiment 1 - Shell variable**

In [ ]:
!MY_VARIABLE="Colab" && echo "Inside this command: $MY_VARIABLE"

Inside this command: Colab


In [ ]:
!echo "In a new shell: $MY_VARIABLE"

In a new shell: 


**Experiment 2 - Python variable**

In [ ]:
my_variable = "Colab"
print(my_variable)

Colab


In [ ]:
print(my_variable)

Colab


**Experiment 3 - Python environment variables**

In [ ]:
import os

os.environ["COURSE_NAME"] = "Cloud Computing"
print(os.environ["COURSE_NAME"])

Cloud Computing


In [ ]:
!echo $COURSE_NAME

Cloud Computing


# Task 1D - A Separate Python Process

In [ ]:
shared_data = [10, 20, 30]
print("Notebook kernel: ", shared_data)

Notebook kernel:  [10, 20, 30]


In [ ]:
!python3 -c "print('External Python process started')"

External Python process started


In [ ]:
!python3 -c "print('shared_data' in globals())"

False


# Task 1E - `%run` vs `!python`

In [ ]:
%%writefile experiment.py

experiment_value = 42
print("Script executed")


Overwriting experiment.py


In [ ]:
%run experiment.py

Script executed


In [ ]:
print(experiment_value)

42


In [ ]:
!python3 experiment.py

Script executed


# Task 1F - Summary
Complete the following table in your notebook.

| Operation                   | Persists across cells? |Explanation                                |
| --------------------------- | :---------------------: | ------------------------------------------ |
| `!cd /tmp` | **False** | Changes only the shell's working directory |
| `%cd /tmp` | **True** | Changes the IPython working directory permanently |
| `!MY_VAR=1` | **False** | Shell-local variable that vanishes when the subshell terminates |
| `my_var = 1` | **True** | Python variable in kernel memory |
| `!mkdir test` | **True*** | Filesystem modification |
| `!touch file.txt` | **True*** | Filesystem modification |
| `!export KEY=value` | **False** | Export applies to that shell process |
| `os.environ["KEY"]="value"` | **True** | Changes the Python process environment |
| `!python3 script.py` | **False** | Runs in a separate process; variables inside script are lost |
| `%run script.py` | **True** | Runs inside IPython; variables/functions load into kernel memory |

*\*Note: Filesystem changes persist until the runtime is reset, disconnected, or manually deleted.*

# Task 2 - Working with Files and Python Code

**2.1 Creating a Python file with `%%writefile**`

In [ ]:
%%writefile math_utils.py

def square_sum(n):
  return sum(i * i for i in range(n))

def factorial(n):
  result = 1
  for i in range(1, n + 1):
    result *= i
  return result


Overwriting math_utils.py


- Verify that the file exists:

In [ ]:
!ls -l math_utils.py

-rw------- 1 root root 150 Aug 19 16:16 math_utils.py


- Inspect the file's contents

In [ ]:
!cat math_utils.py


def square_sum(n):
  return sum(i * i for i in range(n))

def factorial(n):
  result = 1
  for i in range(1, n + 1):
    result *= i
  return result


**2.2 Execute the file with `%run`**

In [ ]:
%run math_utils.py

In [ ]:
print(square_sum(10))
print(factorial(5))

285
120


**2.3 Compare `%run` with ordinary Python import**

In [ ]:
import math_utils

print(math_utils.square_sum(10))

285


| Method |	Purpose |
| ------ | ------- |
|`%run file.py`	| Execute a script through IPython |
|`import module`	| Import a Python module |
|`!python file.py`	| Start a separate Python process and execute the script |


# Task 3 - Measuring Python Performance

3.1 Why timing matters

In [ ]:
%timeit sum(i * i for i in range(1_000_000))

114 ms ± 32.6 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [ ]:
import numpy as np

%timeit np.sum(np.arange(1_000_000, dtype=np.int64) ** 2)

3.43 ms ± 47 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


`NumPy` implementation is much faster because of following reasons:
- **Compiled C Code:** Runs pre-compiled machine code directly, skipping Python's slow step-by-step interpreter.

- **Vectorization:** Computes millions of operations at once using specialized CPU instructions instead of item-by-item for loops.

- **Contiguous Memory:** Stores numbers right next to each other in RAM, making data retrieval virtually instant for the CPU cache.

- **No Object Overhead:** Operates on raw fixed-type numbers rather than heavy, dynamically typed Python integer objects.




# Task 4 - Inspecting the Cloud Hardware

## 4.1 Inspect the CPU

In [ ]:
!lscpu | grep -E "Model name|Socket|Thread|NUMA|CPU\(s\)"

CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0,1


In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       861Mi       8.8Gi       2.0Mi       3.1Gi        11Gi
Swap:             0B          0B          0B


## 4.2 Inspect the operating system

In [ ]:
!cat /etc/os-release

PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy


# Task 5 - GPU Acceleration

## 5.2 Inspect the GPU

In [ ]:
!nvidia-smi

Wed Aug 19 16:18:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

- **GPU Model:** Tesla T4

- **Total GPU Memory:** 15,360 MiB (~15 GB)

- **CUDA Version:** 13.0

- **Current GPU Memory Usage:** 0 MiB / 15360 MiB (0% utilized)

## 5.3 Check GPU availability from PyTorch

In [ ]:
import torch

print("PyTorch version: ", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())

if torch.cuda.is_available():
  print("GPU: ", torch.cuda.get_device_name(0))
  print(
     "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
  )

PyTorch version:  2.11.0+cu128
CUDA available:  True
GPU:  Tesla T4
GPU memory: 14.56 GB


# Task 6 - CPU vs GPU Matrix Multiplication

## 6.2 Benchmark implementation

In [ ]:
import torch
import time

def benchmark_matmul(n=4000):

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Enable a GPU runtime first.")

    print(f"Matrix size: {n} x {n}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    # -------------------------
    # CPU
    # -------------------------
    a_cpu = torch.randn(n, n, dtype=torch.float32)
    b_cpu = torch.randn(n, n, dtype=torch.float32)

    # Warm up CPU execution
    _ = torch.matmul(a_cpu, b_cpu)

    start = time.perf_counter()
    c_cpu = torch.matmul(a_cpu, b_cpu)
    cpu_time = time.perf_counter() - start

    # -------------------------
    # GPU
    # -------------------------
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    # Warm up GPU
    _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()

    start = time.perf_counter()

    c_gpu = torch.matmul(a_gpu, b_gpu)

    # GPU operations are asynchronous.
    # Wait for the GPU before stopping the timer.
    torch.cuda.synchronize()

    gpu_time = time.perf_counter() - start

    speedup = cpu_time / gpu_time

    print(f"CPU time: {cpu_time:.4f} s")
    print(f"GPU time: {gpu_time:.4f} s")
    print(f"GPU speedup: {speedup:.2f}x")

    return cpu_time, gpu_time, speedup

In [ ]:
cpu_time, gpu_time, speedup = benchmark_matmul(4000)

Matrix size: 4000 x 4000
GPU: Tesla T4
CPU time: 1.0145 s
GPU time: 0.0442 s
GPU speedup: 22.94x


# Task 7 - Understanding CUDA Asynchrony

## 7.1 Deliberately perform an incorrect benchmark

In [ ]:
import torch
import time

def benchmark_matmul(n=4000):

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Enable a GPU runtime first.")

    print(f"Matrix size: {n} x {n}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    # -------------------------
    # CPU
    # -------------------------
    a_cpu = torch.randn(n, n, dtype=torch.float32)
    b_cpu = torch.randn(n, n, dtype=torch.float32)

    # Warm up CPU execution
    _ = torch.matmul(a_cpu, b_cpu)

    start = time.perf_counter()

    # -------------------------
    # GPU
    # -------------------------
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    # Warm up GPU
    _ = torch.matmul(a_gpu, b_gpu)

    incorrect_time = time.perf_counter() - start

    print("Without synchronization:", incorrect_time)

benchmark_matmul()

Matrix size: 4000 x 4000
GPU: Tesla T4
Without synchronization: 0.035105085999987296


In [ ]:
import torch
import time

def benchmark_matmul(n=4000):

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available. Enable a GPU runtime first.")

    print(f"Matrix size: {n} x {n}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    # -------------------------
    # CPU
    # -------------------------
    a_cpu = torch.randn(n, n, dtype=torch.float32)
    b_cpu = torch.randn(n, n, dtype=torch.float32)

    # Warm up CPU execution
    _ = torch.matmul(a_cpu, b_cpu)

    start = time.perf_counter()

    # -------------------------
    # GPU
    # -------------------------
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    # Warm up GPU
    _ = torch.matmul(a_gpu, b_gpu)

    torch.cuda.synchronize()

    correct_time = time.perf_counter() - start

    print("With synchronization:", correct_time)

benchmark_matmul()

Matrix size: 4000 x 4000
GPU: Tesla T4
With synchronization: 0.07354268699998556


# Task 8 - Investigate the Effect of Problem Size

In [ ]:
for size in [500, 1000, 2000, 3000]:
    print("\n" + "=" * 50)
    cpu_time, gpu_time, speedup = benchmark_matmul(size)


Matrix size: 500 x 500
GPU: Tesla T4
With synchronization: 0.0012988419999828693

Matrix size: 1000 x 1000
GPU: Tesla T4
With synchronization: 0.005224595000072441

Matrix size: 2000 x 2000
GPU: Tesla T4
With synchronization: 0.0132631079999328

Matrix size: 3000 x 3000
GPU: Tesla T4
With synchronization: 0.034947968999972545


| Matrix size	|   CPU time	|   GPU time	|    GPU speedup |
| ----------- | :---------: | :---------: | :------------: |
| 500 × 500		|
| 1000 × 1000	|
| 2000 × 2000	|
| 3000 × 3000	|

1. Does GPU speedup increase with problem size?
- Yes. Matrix multiplication operations grow at $O(N^3)$ computational complexity while memory requirements grow at $O(N^2)$. As the matrix size $N$ grows larger, compute time quickly dominates, allowing the GPU’s thousands of parallel cores to scale far better than a CPU.

2. Is the GPU always faster?
- No. For very tiny problem sizes (e.g., $10 \times 10$ matrices), a CPU is often faster than a GPU.

3. Why might a small matrix not benefit as much?
- GPU Under-utilization: A Tesla T4 has 2,560 CUDA cores. Small matrix operations do not provide enough compute work to keep all cores busy, leaving most of the GPU idle.

- Low Arithmetic Intensity: The compute work isn't large enough to hide the delays in fetching data from VRAM.

4. What role might data transfer and kernel-launch overhead play?
- Kernel Launch Overhead: Sending a execution command from the CPU to the GPU takes a fixed amount of setup time (a few microseconds). For tiny inputs, launching the job takes longer than calculating the result.

- PCIe Bus Transfer: Copying data from host RAM over the PCIe bus into GPU memory introduces high latency. If transfer time exceeds computation time, the GPU loses its speed advantage.

# Task 9 - Verify CPU and GPU Results

In [ ]:
import torch

n = 4000

# 1. Create tensors on CPU
a_cpu = torch.randn(n, n, dtype=torch.float32)
b_cpu = torch.randn(n, n, dtype=torch.float32)

# 2. Multiply on CPU
c_cpu = torch.matmul(a_cpu, b_cpu)

# 3. Move inputs to GPU and multiply
a_gpu = a_cpu.to("cuda")
b_gpu = b_cpu.to("cuda")
c_gpu = torch.matmul(a_gpu, b_gpu)

# 4. Compare CPU and GPU results
difference = torch.max(torch.abs(c_cpu - c_gpu.cpu()))

print("Maximum absolute difference:", difference.item())

Maximum absolute difference: 0.0009765625
